# Head analysis notebook

Loads the most recent prompt from the dashboard cache and reproduces the same four figures.
Set `LAYER_IDX` and `HEAD_IDX` below to explore a specific head.

## Imports

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from gemma_explore.qwen_core import DEFAULT_MODEL, load_bundle
from gemma_explore.qwen_cache import ensure_scores, ensure_frequency_scores, load_cache
from gemma_explore.qwen_viz import (
    plot_pos_sym_heatmaps,
    plot_heads_scatter,
    plot_all_attention_heads,
    plot_head_block_dynamics,
    plot_frequency_analysis,
)

## Configuration

Edit these two variables to choose which layer/head to inspect.

In [ ]:
LAYER_IDX = 0
HEAD_IDX  = 7

# Score computation params — must match what the dashboard used
N_BLOCKS = 16
TAU      = 0.1

## Load dashboard cache

Reads the most recently added prompt from `data/cache/prompts_registry.json`.

In [ ]:
REGISTRY_PATH = PROJECT_ROOT / "data" / "cache" / "prompts_registry.json"
entries = json.loads(REGISTRY_PATH.read_text())
if not entries:
    raise RuntimeError("Registry is empty — run a prompt in the dashboard first.")

entry = entries[-1]  # most recently added
cache_path = PROJECT_ROOT / entry["cache_path"]
apply_chat_template = entry["apply_chat_template"]

print(f"Prompt : {entry['prompt_text']!r}")
print(f"Cache  : {cache_path}")

cache  = load_cache(cache_path)
bundle = load_bundle(DEFAULT_MODEL)
nl, nh = bundle.num_layers, bundle.num_heads
print(f"Model  : {nl} layers, {nh} heads")

## Compute scores (cached after first run)

In [ ]:
scores = ensure_scores(
    cache, bundle=bundle, prompt_id=0,
    n_blocks=N_BLOCKS, tau=TAU,
    apply_chat_template=apply_chat_template,
)
freq_scores = ensure_frequency_scores(
    cache, bundle=bundle, prompt_id=0,
    n_blocks=N_BLOCKS, tau=TAU,
    apply_chat_template=apply_chat_template,
)

## Fig 1 — Pos/sym scores across all layers and heads

In [ ]:
plot_pos_sym_heatmaps(scores, num_layers=nl, num_heads=nh);

## Fig 2 — Heads scatter (pos vs sym)

In [ ]:
plot_heads_scatter(scores, num_layers=nl, num_heads=nh);

## Fig 3 — All attention heads for layer `LAYER_IDX`

In [ ]:
plot_all_attention_heads(cache, prompt_id=0, layer_idx=LAYER_IDX);

## Fig 4 — Hidden-state block dynamics for (`LAYER_IDX`, `HEAD_IDX`)

In [ ]:
plot_head_block_dynamics(cache, prompt_id=0, layer_idx=LAYER_IDX, head_idx=HEAD_IDX);

## Fig 5 — Frequency analysis for (`LAYER_IDX`, `HEAD_IDX`)

In [ ]:
plot_frequency_analysis(freq_scores, layer_idx=LAYER_IDX, head_idx=HEAD_IDX);